In [362]:
# Cellule 1 : Imports et configuration
%load_ext autoreload
%autoreload 2

import pandas as pd

# Import du processeur de production spécialisé
from tools.OI_class_OP import OI_ProductionProcessor
from tools.OI_Dashboard import ProductionDashboard
from tools.OI_Dashboard_v2 import AjouterVisualisationsAvancees


# Configuration de l'affichage pour voir toutes les colonnes
pd.set_option('display.max_columns', None)

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [363]:
# Cellule 2 : Définition des métadonnées de tags API
tags = [
    {'tag':'WQ33222VA', 'nom':'ester_cons','info':'Totalisation du Peson Acetate/Propionate', 'agg': 'FIRST' },
    {'tag':'NOP_ESTERS', 'nom':'ester_nop','info':"Nombres des opérations d'esters",'agg':'FIRST' },
    {'tag':'3340_type', 'nom':'A/P','info':'Acetate ou Propionate','agg':'FIRST' },
    {'tag':'CTY_ACV43A_Teneur Vit. A (UV)', 'nom':'Acetate_UV','info':'ACV43A teneur en acetate', 'agg': 'MEAN'},
    {'tag':'CTY_A3340FGB_Teneur arr. Vit. A (UV)', 'nom':'Propionate_UV', 'info':'A3340FGB teneur en propionate', 'agg': 'MEAN'},
    {'tag':'PU3310VA_Sign','nom':'PU3310','info':'signature du PU3310','agg':'FIRST'},
    {'tag':'PU3320VA_Sign','nom':'PU3320','info':'signature du PU3320','agg':'FIRST'},
    {'tag':'PU3340VA_Sign','nom':'PU3340','info':'signature du PU3340','agg':'FIRST'},
    {'tag':'FQ32202VA_UV','nom':'Hexane','info':'VA diluée dans de l\'hexane', 'agg': 'FIRST'},
    {'tag':'CTY_A3230A_Teneur en rétinol', 'nom':'Retinol_UV','info':'A3230A teneur en rétinol lavé', 'agg': 'MEAN'},
    {'tag':'LI33203VA','nom':'R33020','info':'niveau du R33020', 'agg': 'FIRST'},
    {'tag':'LI33218VA','nom':'R33022','info':'niveau du R33022', 'agg': 'FIRST'},
    {'tag':'LI33225VA','nom':'R33061','info':'niveau du R33061', 'agg': 'FIRST'},
    {'tag':'WI33222VA','nom':'R33060','info':'peson du R33060', 'agg': 'FIRST'},
    {'tag':'FQ32202VA','nom':'retinol_cons','info':'peson du R33060', 'agg': 'FIRST'},
    {'tag':'LI32209VA','nom':'R32031','info':'niveau du R32031', 'agg': 'FIRST'},
]

In [364]:
# Cellule 3 : Variable Produits unifiée (Acetate & Propionate)
# Plus aucune distinction batch / continu pour le calcul global du stock d'un produit.
produits = [
    {
        'nom': 'Ester',
        'conso': {
            'value': 'ester_cons',
            'scale': 1e-9,
            'type':'A/P',
            'uv': ['Acetate_UV', 'Propionate_UV'],
        },
        'CMJ': 9.5,
        'NOP': {
            'value':'NOP_ESTERS',
            'scale': 1,
            'type': None,
        },
        'stock': [
            # Batchs
            {'pu': 'PU3310', 'in': 540, 'out': 710, 'value': 'Hexane', 'uv': 'Retinol_UV', 'scale': 1/(825 * 100.)},
            {'pu': 'PU3310', 'in': 710, 'out': 2320, 'value': None, 'uv': None, 'scale': 2.71},
            {'pu': 'PU3320', 'in': 430, 'out': 20210, 'value': None, 'uv': None, 'scale': 2.71},
            # Continus
            {'pu': None, 'value': 'R33020', 'min': 14, 'epalage': [[28.62, 21.22, 3.866, -0.0983], [-228.84, 74.557]], 'uv': 'Retinol_UV', 'scale': 1/100., 'cond':'A/P', 'val_cond':[1/344,1/359]},
            {'pu': None, 'value': 'R33022', 'min': 14, 'epalage': [[6.25, 5.4525, 0.898, -0.0256], [-35, 97, 16.039]], 'uv': 'Retinol_UV', 'scale': 1/100., 'cond':'A/P', 'val_cond':[1/344,1/359]},
            {'pu': None, 'value': 'R33061', 'min': 18, 'epalage': [[0.69, 0.72, 0.296, -0.0059], [-30.03, 5.836]], 'uv': None, 'scale': 0.95 , 'cond':'A/P', 'val_cond':[1/344,1/359]},
            {'pu': 'PU3340', 'in': 0, 'out': 710, 'value': 'R33060', 'uv': None, 'scale': 0.95, 'cond': 'A/P', 'val_cond': [1/344, 1/359]}
        ]
    },
    {
        'nom': 'Retinol',
        'conso': {
            'value':'retinol_cons',
            'scale': 1e-5, # Ajustement de l'échelle pour le retinol g et analyse en %
            'uv': ['Retinol_UV','Retinol_UV'],
        },
        'CMJ': 1,
        'stock': [
            # Batchs
            {'pu': None, 'value': 'R32031', 'uv': 'Retinol_UV', 'scale': 1/(825 * 100.)},
        ]
    }
]

In [365]:
# Cellule 4 : Initialisation du processeur de production spécialisé
processor = OI_ProductionProcessor(
    url_base = 'https://oianalytics-100.optimistik.fr/api/oianalytics/time-values/query?',
    start = '2026-01-01',
    end = '2026-12-31',
    tags_metadata = tags,
    produits = produits,
    interval = 'PT15M',
    verbose = False
)

In [366]:
(processor.agg_mapping)

{'WQ33222VA': 'FIRST',
 'NOP_ESTERS': 'FIRST',
 '3340_type': 'FIRST',
 'CTY_ACV43A_Teneur Vit. A (UV)': 'MEAN',
 'CTY_A3340FGB_Teneur arr. Vit. A (UV)': 'MEAN',
 'PU3310VA_Sign': 'FIRST',
 'PU3320VA_Sign': 'FIRST',
 'PU3340VA_Sign': 'FIRST',
 'FQ32202VA_UV': 'FIRST',
 'CTY_A3230A_Teneur en rétinol': 'MEAN',
 'LI33203VA': 'FIRST',
 'LI33218VA': 'FIRST',
 'LI33225VA': 'FIRST',
 'WI33222VA': 'FIRST',
 'FQ32202VA': 'FIRST',
 'LI32209VA': 'FIRST'}

In [367]:
# Cellule 5 : Téléchargement et calcul automatique des bilans par produit
processor.merge()
processor.compute_production_balance()

# Visualisation des premières lignes calculées
processor.data.describe()

,ester_cons,ester_nop,A/P,Acetate_UV,Propionate_UV,PU3310,PU3320,PU3340,Hexane,Retinol_UV,R33020,R33022,R33061,R33060,retinol_cons,R32031,consommation_Ester,stock_Ester_tmp_PU3310_Hexane_0,stock_Ester_tmp_PU3310_None_1,stock_Ester_tmp_PU3320_None_2,stock_Ester_tmp_None_R33020_3,stock_Ester_tmp_None_R33022_4,stock_Ester_tmp_None_R33061_5,stock_Ester_tmp_PU3340_R33060_6,stock_Ester,consommation_Retinol,stock_Retinol_tmp_None_R32031_0,stock_Retinol,conso_delta_Ester,delta_stock_Ester,production_Ester,conso_delta_Retinol,delta_stock_Retinol,production_Retinol
count,1.553900e+04,15539.000000,15539.000000,1.553900e+04,1.553900e+04,15531.000000,15533.000000,15533.000000,15539.000000,15539.000000,15539.000000,15539.000000,15539.000000,15539.000000,15539.000000,15539.000000,15539.000000,15539.000000,15539.000000,15539.000000,15539.000000,15539.000000,15539.000000,15539.000000,15539.000000,15539.000000,15539.000000,15539.000000,15539.000000,15539.000000,15539.000000,15539.000000,15539.000000,15539.000000
mean,2.468375e+06,1976.242044,0.071299,2.226498e+06,2.275237e+06,1029.314919,963.099079,460.814395,2598.522086,20.778115,46.218910,4.963778,7.264254,689.137428,484405.945389,47.661004,550.812980,0.029439,1.407233,1.692202,1.937605,0.033640,0.077274,1.420373,6.597766,432354.177103,0.012002,0.012002,550.812980,1.011479,551.824459,432354.177103,0.000447,432354.177551
std,1.551319e+05,127.232912,0.257288,3.742800e+04,5.459055e+04,623.524613,553.781784,119.325784,1884.446710,0.360469,20.419832,2.472635,11.536614,352.294679,257103.878565,21.142737,347.313570,0.144626,1.354036,1.312414,0.914148,0.016196,0.168272,0.951366,2.165344,495014.093448,0.005326,0.005326,347.313570,2.165344,347.918186,495014.093448,0.005326,495014.093693
min,2.221310e+06,1774.000000,0.000000,2.078915e+06,2.188000e+06,100.000000,100.000000,100.000000,0.000000,20.000000,-8.142390,-0.695313,-0.418438,4.689820,640.456000,-1.303840,0.000000,0.000000,0.000000,0.000000,0.000805,0.001665,0.001218,0.000000,0.707269,0.000000,-0.000329,-0.000329,0.000000,-4.879019,-0.851652,0.000000,-0.011884,0.000000
25%,2.319790e+06,1855.000000,0.000000,2.221983e+06,2.242000e+06,400.000000,400.000000,410.000000,0.000000,20.600000,24.367650,3.079280,2.244280,426.223000,263744.000000,39.460400,217.913533,0.000000,0.000000,0.000000,0.959168,0.018345,0.010275,0.691550,5.483568,69.597257,0.009909,0.009909,217.913533,-0.102720,220.231289,69.597257,-0.001646,69.604844
50%,2.455020e+06,1964.000000,0.000000,2.231260e+06,2.297000e+06,1310.000000,1110.000000,410.000000,3961.130000,20.800000,46.040000,5.999090,3.031790,683.930000,465362.000000,45.080600,519.090053,0.000000,2.710000,2.710000,1.925621,0.039587,0.014982,1.506297,6.969960,162.692229,0.011505,0.011505,519.090053,1.383673,519.240635,162.692229,-0.000050,162.699554
75%,2.601360e+06,2084.500000,0.000000,2.243007e+06,2.314000e+06,1310.000000,1555.000000,410.000000,4043.100000,21.200000,65.125950,6.574220,4.033205,947.499000,711261.000000,67.148400,847.942761,0.000000,2.710000,2.710000,2.781892,0.044309,0.022108,2.195044,8.052655,999175.920979,0.016841,0.016841,847.942761,2.466367,848.903588,999175.920979,0.005286,999175.918838
max,2.755760e+06,2212.000000,1.000000,2.325113e+06,2.626000e+06,2340.000000,2250.000000,810.000000,4836.070000,21.200000,91.590400,12.284600,73.884600,1323.150000,999754.000000,79.734400,1195.501077,1.207552,2.710000,2.710000,3.933059,0.099400,1.107856,3.581113,12.323393,999279.640346,0.019909,0.019909,1195.501077,6.737105,1193.526995,999279.640346,0.008354,999279.646898


In [368]:
# ✨ INITIALISER LE DASHBOARD ✨
dashboard = ProductionDashboard(processor)
AjouterVisualisationsAvancees(dashboard) 

print("\n✓ Dashboard prêt pour utilisation!")
#dashboard.resume_complet()


✓ Dashboard initialisé
  Produits: Ester, Retinol
  Période: 2026-01-01 → 2026-06-12
✅ Visualisations avancées ajoutées au dashboard!

   Nouvelles méthodes disponibles:
   • dashboard.plot_histogramme_tous_produits(mois=3)
   • dashboard.plot_waterfall_mois(mois=3)
   • dashboard.plot_histogramme_jours_mois_v1(mois=3, nom_produit='Ester')
   • dashboard.plot_histogramme_jours_mois_v2(mois=3, nom_produit='Ester')


✓ Dashboard prêt pour utilisation!


In [369]:
# Voir l'évolution temporelle d'un produit
dashboard.afficher_bilan_produit('Ester')


In [370]:
dashboard.plot_histogramme_jours_mois_v2(mois=6, annea=2026, nom_produit='Ester', std=2)

____________________________________________________________
BILAN JOURNALIER - 01 JUIN 2026 (Terminé)
Période : du 01/06/2026 à 02:00 au 02/06/2026 à 02:00
____________________________________________________________
PRODUIT : ESTER
------------------------------------------------------------
  Consommation (Ester)   : 8.3173
  Variation de Stock     : 2.4726
  Stock Entrée.          : -0.2890
  Stock Sortie.          : 2.1836
  Production             : 10.7899
____________________________________________________________
PRODUIT : RETINOL
------------------------------------------------------------
  Consommation (Ester)   : 3.3908
  Variation de Stock     : 0.0057
  Stock Entrée.          : -0.0004
  Stock Sortie.          : 0.0052
  Production             : 3.3964
____________________________________________________________
____________________________________________________________
____________________________________________________________
BILAN JOURNALIER - 02 JUIN 2026 (Termin


  📊 STATISTIQUES - PRODUCTION ESTER PAR JOUR - JUNE 2026
Nombre de jours complets:           12
Production moyenne:                 8.12
Production min/max:                 0.06 / 14.13
Écart-type:                         3.57
Coefficient de variation:           44.0%
Production totale mois:             97.46
TRS mois: (jours complets)          8.12
Jours au-dessus de la moyenne:      8 / 12

🎯 RATIO oee (Production / CMJ en %):
  CMJ (Cible Journalière):            9.50
  oee Moyen (par jour):               85.5%
  oee Min/Max (par jour):             0.6% / 148.7%
  oee Cumulé à date:                  85.5%
  Jours > 100%:                       4 / 12

  Détail oee par jour:
    J01:  113.6% ✅
    J02:   85.6% ⚠️ 
    J03:   94.0% ⚠️ 
    J04:  113.8% ✅
    J05:   77.9% ❌
    J06:  148.7% ✅
    J07:   99.1% ⚠️ 
    J08:   74.7% ❌
    J09:  104.3% ✅
    J10:   86.2% ⚠️ 
    J11:   27.3% ❌
    J12:    0.6% ❌



In [371]:
debut = '06-06-2026 02:00:00'
fin = '06-07-2026 02:00:00'
NOP = processor.data.loc[[fin],['ester_nop']].values[0] - processor.data.loc[[debut],['ester_nop']].values[0]
print('Ester NOP   : ', NOP, NOP * 2.71)
print('Ester peson : ',processor.data.loc[[fin],['ester_cons']].values[0] - processor.data.loc[[debut],['ester_cons']].values[0] )
print('Ester titre : ',processor.data.loc[[fin],['consommation_Ester']].values[0] - processor.data.loc[[debut],['consommation_Ester']].values[0])
print('Ester Stock : ',processor.data.loc[[fin],['stock_Ester']].values[0], processor.data.loc[[debut],['stock_Ester']].values[0] )
print('Ester Delta : ',processor.data.loc[[fin],['stock_Ester']].values[0] - processor.data.loc[[debut],['stock_Ester']].values[0])
print('Ester conso : ',processor.data.loc[[fin],['conso_delta_Ester']].values[0]- processor.data.loc[[debut],['conso_delta_Ester']].values[0] )
print('Ester prod  : ',processor.data.loc[[fin],['production_Ester']].values[0]- processor.data.loc[[debut],['production_Ester']].values[0] )
processor.data[debut:fin]

Ester NOP   :  [3.] [8.13]
Ester peson :  [4290.]
Ester titre :  [9.52241862]
Ester Stock :  [11.36040689] [6.75235152]
Ester Delta :  [4.60805538]
Ester conso :  [9.52241862]
Ester prod  :  [14.130474]


,ester_cons,ester_nop,A/P,Acetate_UV,Propionate_UV,PU3310,PU3320,PU3340,Hexane,Retinol_UV,R33020,R33022,R33061,R33060,retinol_cons,R32031,consommation_Ester,stock_Ester_tmp_PU3310_Hexane_0,stock_Ester_tmp_PU3310_None_1,stock_Ester_tmp_PU3320_None_2,stock_Ester_tmp_None_R33020_3,stock_Ester_tmp_None_R33022_4,stock_Ester_tmp_None_R33061_5,stock_Ester_tmp_PU3340_R33060_6,stock_Ester,consommation_Retinol,stock_Retinol_tmp_None_R32031_0,stock_Retinol,conso_delta_Ester,delta_stock_Ester,production_Ester,conso_delta_Retinol,delta_stock_Retinol,production_Retinol
timestamp,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
2026-06-06 02:00:00+00:00,2730780.0,2192.0,0.0,2219678.0,2188000.0,2320.0,430.0,410.0,4042.42,20.6,64.3027,0.32374,3.30051,465.954,787465.0,37.61880,1139.761947,0.000000,0.00,2.71,2.733917,0.004856,0.016787,1.286792,6.752352,999264.307123,0.009393,0.009393,1139.761947,1.166064,1140.928011,999264.307123,-0.002162,999264.304961
2026-06-06 02:15:00+00:00,2730780.0,2192.0,0.0,2219678.0,2188000.0,2320.0,500.0,410.0,4042.42,20.6,76.2267,6.26884,2.86202,474.979,787465.0,24.68000,1139.761947,0.000000,0.00,2.71,3.266294,0.041568,0.013910,1.311715,7.343487,999264.307123,0.006163,0.006163,1139.761947,1.757200,1141.519147,999264.307123,-0.005393,999264.301731
2026-06-06 02:30:00+00:00,2730780.0,2192.0,0.0,2219678.0,2188000.0,2320.0,660.0,410.0,4042.42,20.6,80.8983,6.26082,2.97003,519.889,787465.0,12.82440,1139.761947,0.000000,0.00,2.71,3.474869,0.041502,0.014595,1.435740,7.676706,999264.307123,0.003202,0.003202,1139.761947,2.090419,1141.852366,999264.307123,-0.008353,999264.298770
2026-06-06 02:45:00+00:00,2730780.0,2192.0,0.0,2219678.0,2188000.0,540.0,1110.0,410.0,1651.78,20.6,81.8018,6.25534,3.13658,564.513,789110.0,33.95010,1139.761947,0.412444,0.00,2.71,3.515208,0.041457,0.015682,1.558975,8.253766,999264.645993,0.008477,0.008477,1139.761947,2.667479,1142.429426,999264.645993,-0.003078,999264.642915
2026-06-06 03:00:00+00:00,2730780.0,2192.0,0.0,2219678.0,2188000.0,600.0,1120.0,410.0,3985.68,20.6,79.8503,6.28529,3.25815,608.403,791446.0,64.80490,1139.761947,0.995212,0.00,2.71,3.428079,0.041703,0.016498,1.680183,8.871674,999265.127209,0.016182,0.016182,1139.761947,3.285387,1143.047334,999265.127209,0.004626,999265.131836
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2026-06-07 01:00:00+00:00,2734530.0,2195.0,0.0,2219678.0,2188000.0,1310.0,1120.0,410.0,4072.03,20.6,67.4733,6.68750,3.52093,914.048,804186.0,27.40470,1148.085740,0.000000,2.71,2.71,2.875477,0.045043,0.018329,2.524260,10.883109,999267.751649,0.006843,0.006843,1148.085740,5.296822,1153.382562,999267.751649,-0.004712,999267.746937
2026-06-07 01:15:00+00:00,2734530.0,2195.0,0.0,2219678.0,2188000.0,1310.0,1120.0,410.0,4072.03,20.6,64.9311,6.65234,3.53424,957.245,804186.0,11.55770,1148.085740,0.000000,2.71,2.71,2.761974,0.044748,0.018424,2.643555,10.888701,999267.751649,0.002886,0.002886,1148.085740,5.302413,1153.388153,999267.751649,-0.008669,999267.742980
2026-06-07 01:30:00+00:00,2734530.0,2195.0,0.0,2219678.0,2188000.0,1310.0,1530.0,410.0,4072.03,20.6,61.8575,6.65234,3.49692,999.811,804186.0,7.97612,1148.085740,0.000000,2.71,2.71,2.624745,0.044748,0.018158,2.761106,10.868757,999267.751649,0.001992,0.001992,1148.085740,5.282470,1153.368210,999267.751649,-0.009564,999267.742086


In [372]:
processor.calcul_cumul_journalier(1,6,2026,"Ester")

____________________________________________________________
BILAN JOURNALIER - 01 JUIN 2026 (Terminé)
Période : du 01/06/2026 à 02:00 au 02/06/2026 à 02:00
____________________________________________________________
PRODUIT : ESTER
------------------------------------------------------------
  Consommation (Ester)   : 8.3173
  Variation de Stock     : 2.4726
  Stock Entrée.          : -0.2890
  Stock Sortie.          : 2.1836
  Production             : 10.7899
____________________________________________________________
____________________________________________________________


{'Ester': {'consommation': 8.317302400000017,
  'delta_stock': 2.472643477741661,
  'production': 10.78994587774173}}

In [373]:
dashboard.afficher_bilan_journalier(jour=6, mois=6, annee=2026)

____________________________________________________________
BILAN JOURNALIER - 06 JUIN 2026 (Terminé)
Période : du 06/06/2026 à 02:00 au 07/06/2026 à 02:00
____________________________________________________________
PRODUIT : ESTER
------------------------------------------------------------
  Consommation (Ester)   : 9.5224
  Variation de Stock     : 4.6081
  Stock Entrée.          : 1.1661
  Stock Sortie.          : 5.7741
  Production             : 14.1305
____________________________________________________________
PRODUIT : RETINOL
------------------------------------------------------------
  Consommation (Ester)   : 3.4445
  Variation de Stock     : 0.0018
  Stock Entrée.          : -0.0022
  Stock Sortie.          : -0.0003
  Production             : 3.4464
____________________________________________________________
____________________________________________________________


In [374]:
dashboard.plot_histogramme_annuee(annea=2026, nom_produit='Ester')

____________________________________________________________
BILAN JOURNALIER - 01 JANVIER 2026 (Terminé)
Période : du 01/01/2026 à 02:00 au 02/01/2026 à 01:30
____________________________________________________________
PRODUIT : ESTER
------------------------------------------------------------
  Consommation (Ester)   : 8.2673
  Variation de Stock     : -1.3901
  Stock Entrée.          : 3.0225
  Stock Sortie.          : 1.6324
  Production             : 6.8772
____________________________________________________________
PRODUIT : RETINOL
------------------------------------------------------------
  Consommation (Ester)   : 2.5713
  Variation de Stock     : -0.0014
  Stock Entrée.          : 0.0015
  Stock Sortie.          : 0.0001
  Production             : 2.5700
____________________________________________________________
____________________________________________________________
____________________________________________________________
BILAN JOURNALIER - 02 JANVIER 2026 (T


  📊 RÉSUMÉ ANNUEL 2026 - Ester
Mois         Jours    Prod Total      Prod Moy        oee Moy      oee Cumul   
----------------------------------------------------------------------------------------------------
January      29       185.12          6.38            67.2        % 67.2        %
February     26       188.52          7.25            76.3        % 76.3        %
March        28       210.44          7.52            79.1        % 79.1        %
April        30       246.96          8.23            86.7        % 86.7        %
May          31       261.33          8.43            88.7        % 88.7        %
June         12       97.46           8.12            85.5        % 85.5        %



In [375]:
dashboard.plot_histogramme_annee_complet(annea=2026, nom_produit='Ester', std=2)

____________________________________________________________
BILAN JOURNALIER - 01 JANVIER 2026 (Terminé)
Période : du 01/01/2026 à 02:00 au 02/01/2026 à 01:30
____________________________________________________________
PRODUIT : ESTER
------------------------------------------------------------
  Consommation (Ester)   : 8.2673
  Variation de Stock     : -1.3901
  Stock Entrée.          : 3.0225
  Stock Sortie.          : 1.6324
  Production             : 6.8772
____________________________________________________________
PRODUIT : RETINOL
------------------------------------------------------------
  Consommation (Ester)   : 2.5713
  Variation de Stock     : -0.0014
  Stock Entrée.          : 0.0015
  Stock Sortie.          : 0.0001
  Production             : 2.5700
____________________________________________________________
____________________________________________________________
____________________________________________________________
BILAN JOURNALIER - 02 JANVIER 2026 (T


  📊 RÉSUMÉ ANNUEL 2026 - Ester
Nombre de jours avec données:       156/365
Production totale année:            1189.83
Production moyenne (jours actifs):  7.63
Production min/max:                 0.03 / 14.13
Écart-type:                         2.98

🎯 RATIO oee:
  CMJ (Cible Journalière):            9.50
  oee Moyen (par jour):               80.3%
  oee Cumulé à date (fin d'année):    34.3%
  Jours > 100% (surproduction):       35 / 156



In [376]:
processor.plot_simple_tag('Hexane')